In [1]:
import os
from pyspark.sql import SparkSession

base_dir = r"E:\BIGDATA\data\parquet"
temp_dir = os.path.join(base_dir, "spark_temp")
os.makedirs(temp_dir, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("PCA_feature_21")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.local.dir", temp_dir)
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)
spark

In [2]:
parquet_path = r"E:\BIGDATA\data\parquet\cic_ddos_2019_10gb_cleaned.parquet"

df_raw = spark.read.parquet(parquet_path)
df_raw.printSchema()

root
 |-- label: string (nullable = true)
 |-- source_port: integer (nullable = true)
 |-- destination_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- total_fwd_packets: integer (nullable = true)
 |-- total_backward_packets: integer (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- total_length_of_bwd_packets: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: double (nullable = true)
 |-- bwd_packet_length_min: double (nullable = true)
 |-- bwd_packet_length_mean: double (nullable = true)
 |-- bwd_packet_length_std: double (nullable = true)
 |-- flow_bytes_per_s: double (nullable = true)
 |-- flow_packets_per_s: double (nullabl

In [3]:
from pyspark.sql import functions as F


df = df_raw


print("Rows:", df.count())
df.show(5, truncate=False)

Rows: 24207109
+----------+-----------+----------------+--------+--------------------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+----------------+------------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+-----------------+-------------------+--------------------+--------------------+-------------------+-------------------+-----------------+---------

In [5]:
exclude_cols = ["label", "timestamp", "simillarhttp"]

feature_cols = [
    c for c, t in df_raw.dtypes
    if c not in exclude_cols and t != "string"
]
print("Feature columns:", feature_cols)

Feature columns: ['source_port', 'destination_port', 'protocol', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes_per_s', 'flow_packets_per_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packets_per_s', 'bwd_packets_per_s', 'min_packet_length', 'max_packet_length', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'syn_flag_count', 'rst_flag_count', 'ack_flag_count', 'urg_flag_count', 'cwe_flag_count', 'down_per_up_ratio', 'average_packet_size'

In [6]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
df_vec = assembler.transform(df_raw).select(
    "features",
    "label"
)


In [7]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df_vec)

df_scaled = scaler_model.transform(df_vec).select(
    "scaled_features",
    "label"
)



# K = 21

In [8]:
pca_k21 = PCA(
    k=21,
    inputCol="scaled_features",
    outputCol="pca_k21"
)
pca_k21_model = pca_k21.fit(df_scaled)

df_pca_21 = pca_k21_model.transform(df_scaled).select(
    "pca_k21",
    "label"
)


In [11]:
df_pca_21.show(3, truncate=False)


+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|pca_k21                                                                                                                                                                                                                                                                                                                                                                                                                                   |label     |
+-----------------------------------------------------------------------------------------------------------------------

In [13]:
df_pca_21.count(), df_pca_21.columns, len(df_pca_21.first()["pca_k21"])

(24207109, ['pca_k21', 'label'], 21)

In [12]:
output_path = r"E:\BIGDATA\data\parquet\cic_ddos_pca_21.parquet"

df_pca_21.select("pca_k21", "label").write.mode("overwrite").parquet(output_path)



# K = 25

In [14]:
pca_k25 = PCA(
    k=25,
    inputCol="scaled_features",
    outputCol="pca_k25"
)
pca_k25_model = pca_k25.fit(df_scaled)
df_pca_25 = pca_k25_model.transform(df_scaled).select(
    "pca_k25",
    "label"
)


In [15]:
df_pca_25.count(), df_pca_25.columns, len(df_pca_25.first()["pca_k25"])

(24207109, ['pca_k25', 'label'], 25)

In [16]:
output_path = r"E:\BIGDATA\data\parquet\cic_ddos_pca_25.parquet"

df_pca_25.select("pca_k25", "label").write.mode("overwrite").parquet(output_path)

